# Set up library

In [69]:
!pip install keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 8.8 MB/s eta 0:00:00


In [70]:
import numpy as np
import pandas as pd
import librosa
from pathlib import Path
import matplotlib.pyplot as plt
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.callbacks import ModelCheckpoint,  EarlyStopping
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Dense, Dropout, Flatten, BatchNormalization
from sklearn.model_selection import train_test_split,RandomizedSearchCV
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import LabelEncoder
import keras_tuner as kt
import h5py
from sklearn.utils.class_weight import compute_class_weight
import os





# Mount Drive

In [71]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#Data Module

In [72]:
from os.path import isfile
class DataModule:

    def __init__(self):
        self.data_set_dic = {}

    def __extract_dft(self, window):
        eps = 1e-10
        spectrum = np.abs(np.fft.rfft(window))
        spectrum = spectrum / np.max(spectrum + eps)
        return spectrum

    def __suppress_low_amplitudes(self, spec, threshold_db=50):
        I_threshold = 10 ** (-threshold_db / 20)
        spec[spec < I_threshold] *= 0.2
        return spec
    def __get_spectrum_of_window(self, window):

        spectrum = self.__extract_dft(window)

        #preprocessing
        standard_spectrum = self.__suppress_low_amplitudes(spectrum)

        return standard_spectrum

    def process_one_file_audio(
        self,
        audio_path,
        csv_path,
        sample_rate=48000,
        window_size=0.01
    ):
        #load audio
        audio, sr = librosa.load(audio_path, sr=sample_rate, mono=True)

        samples_per_window = int(sample_rate * window_size)

        # Load CSV
        df = pd.read_csv(csv_path)
        X = []
        y = []

        # preparing data
        for _, row in df.iterrows():
            start_sample = int(row["start_time"] * sample_rate)
            end_sample = start_sample + samples_per_window

            if end_sample > len(audio):
                continue

            window = audio[start_sample:end_sample]
            spectrum = self.__get_spectrum_of_window(window)

            X.append(spectrum)
            y.append(row["label"])

        # reducing silence sample
        silence_label = "silence"
        flag = True
        start_index = 0
        end_index = len(y) - 1
        max_silence_label = 3
        X_reduced = []
        y_reduced = []
        while flag:

            if y[start_index] == silence_label:
                start_index += 1

            if y[end_index] == silence_label:
                end_index -= 1

            if y[start_index] != silence_label and y[end_index] != silence_label:
                X_reduced = X[start_index:end_index + 1]
                y_reduced = y[start_index:end_index + 1]


                num_start_silence = start_index - 0
                num_end_silence = len(y) - 1 - end_index

                if num_start_silence > max_silence_label:
                    num_start_silence = max_silence_label

                if num_end_silence > max_silence_label:
                    num_end_silence = max_silence_label

                for i in np.arange(num_start_silence):
                    X_reduced.append(X[i])
                    y_reduced.append(y[i])

                for i in np.arange(num_end_silence):
                    end_index = len(y) - 1 - i
                    X_reduced.append(X[end_index])
                    y_reduced.append(y[end_index])

                flag = False



        return np.array(X_reduced), np.array(y_reduced)

    def build_dataset(self):

        # read data from drive
        audio_data_set_path = "/content/drive/MyDrive/Qeej_Hmong_Dataset/Data/audio"
        audio_folder_name = "audio"
        labels_folder_name = "labels"

        if os.path.exists(audio_data_set_path):
          for root, dirs, files in os.walk(audio_data_set_path):
            for file in files:
              if file.endswith(".wav"):
                audio_file_path = os.path.join(root, file)
                labels_file_name = file.replace(".wav", ".csv")
                labels_file_root = root.replace(audio_folder_name, labels_folder_name)
                #check if specified audio has corresponded labels file
                labels_path = os.path.join(labels_file_root, labels_file_name)
                # print("check audio path and labels path: ", audio_file_path, labels_path)
                if os.path.exists(labels_path):
                    if os.path.exists(labels_path):
                      self.data_set_dic[audio_file_path] = labels_path
                    else:
                      print(labels_path, "is not a file")
                else:
                  print("Cant find file at", labels_path)

        else:
          print("Cant find folder at /content/drive/MyDrive/Qeej_Hmong_Dataset/Data/audio")

        X_all = []
        y_all = []

        for audio_file_path, labels_file_path in self.data_set_dic.items():
          #  print ("check audio path and corresponding label path: ", audio_file_path, labels_file_path)
           X, y = self.process_one_file_audio(audio_file_path, labels_file_path)
           X_all.append(X)
           y_all.append(y)

        X_all = np.vstack(X_all)
        y_all = np.hstack(y_all)

        return X_all, y_all

#Main

## Set up HDF5 File and labeling Data

In [73]:
HDF5_file_path = "/content/drive/MyDrive/Qeej_Hmong_Dataset/HDF5"
X = None
y = None

if os.path.exists(HDF5_file_path):
    data_set_path = HDF5_file_path + "/Qeej_Hmong_Features.h5"
    if os.path.exists(data_set_path):
      print("Retrieving data from drive with path:", data_set_path)
      with h5py.File(data_set_path, "r") as f:
        X = f["features"][:]
        y = f["labels"][:]

    else:
        print("set up data")
        dataModule = DataModule()
        X, y = dataModule.build_dataset()
        print("check len X:", len(X))
        print("check len y:", len(y))
        with h5py.File(data_set_path, "w") as f:
          f.create_dataset("features", data=X)
          f.create_dataset("labels", data=y.tolist())

Retrieving data from drive with path: /content/drive/MyDrive/Qeej_Hmong_Dataset/HDF5/Qeej_Hmong_Features.h5


##label encoding

In [74]:
le = LabelEncoder()
y_int = le.fit_transform(y)
y_cat = to_categorical(y_int)

class_names = le.classes_
num_classes = y_cat.shape[1]


## Set up Training/Validation/Testing Data

In [75]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_cat, test_size=0.3, stratify=y_int)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.1)

y_train_int = np.argmax(y_train, axis=1)
print("check y train int:", y_train_int)


check y train int: [56 22 11 ... 18 31  1]


## Set up Class Weights

In [76]:
unique, counts = np.unique(y_train_int, return_counts=True)

for label, count in zip(unique, counts):
    print(f"Class {label}: {count} samples")

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train_int),
    y=y_train_int
)

class_weight_dict = dict(enumerate(class_weights))

print("check class weight dict:", class_weight_dict)


Class 0: 2946 samples
Class 1: 2616 samples
Class 2: 2365 samples
Class 3: 2208 samples
Class 4: 2301 samples
Class 5: 2036 samples
Class 6: 1965 samples
Class 7: 2164 samples
Class 8: 1954 samples
Class 9: 2181 samples
Class 10: 2439 samples
Class 11: 2300 samples
Class 12: 2045 samples
Class 13: 2252 samples
Class 14: 2193 samples
Class 15: 2097 samples
Class 16: 2224 samples
Class 17: 2612 samples
Class 18: 2519 samples
Class 19: 2213 samples
Class 20: 2076 samples
Class 21: 2263 samples
Class 22: 2164 samples
Class 23: 2106 samples
Class 24: 2092 samples
Class 25: 2660 samples
Class 26: 2502 samples
Class 27: 2243 samples
Class 28: 2423 samples
Class 29: 2729 samples
Class 30: 2166 samples
Class 31: 2531 samples
Class 32: 1783 samples
Class 33: 2802 samples
Class 34: 2439 samples
Class 35: 2156 samples
Class 36: 2208 samples
Class 37: 2123 samples
Class 38: 2122 samples
Class 39: 2015 samples
Class 40: 2215 samples
Class 41: 2180 samples
Class 42: 2514 samples
Class 43: 2148 sample

## Set up params grid for random search to choose hyper paramenters

In [77]:
def build_model(hp):

    Lc = hp.Choice("Lc", [2,3,4])
    f1 = hp.Choice("f1", [16,32])
    f2 = hp.Choice("f2", [32,64,128])
    c = hp.Choice("c", [3,5,7])
    h = hp.Choice("h", [64,128,256])
    do1 = hp.Choice("do1", [0.1,0.25])
    do2 = hp.Choice("do2", [0.25,0.5])

    model = Sequential()

    model.add(
        Conv1D(
            filters=f1,
            kernel_size=c,
            activation="relu",
            input_shape=(X_train.shape[1],1)
        )
    )

    model.add(MaxPooling1D(2))
    model.add(Dropout(do1))

    for i in range(Lc-1):

        model.add(
            Conv1D(
                filters=f2,
                kernel_size=c,
                activation="relu"
            )
        )

        model.add(MaxPooling1D(2))
        model.add(Dropout(do1))

    model.add(Flatten())

    model.add(Dense(h, activation="relu"))
    model.add(Dropout(do2))

    model.add(Dense(num_classes, activation="softmax"))

    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

## Random Search

In [78]:
tuner = kt.RandomSearch(
    build_model,
    objective="val_accuracy",
    max_trials=50,
    executions_per_trial=1
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
tuner.search(
    X_train[...,None],
    y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=512
)

best_hp = tuner.get_best_hyperparameters(1)[0]

print(best_hp.values)

Trial 42 Complete [00h 03m 29s]
val_accuracy: 0.39636266231536865

Best val_accuracy So Far: 0.7319044470787048
Total elapsed time: 02h 40m 14s

Search: Running Trial #43

Value             |Best Value So Far |Hyperparameter
4                 |2                 |Lc
32                |32                |f1
128               |64                |f2
5                 |5                 |c
128               |256               |h
0.25              |0.1               |do1
0.25              |0.25              |do2

Epoch 1/50
236/236 ━━━━━━━━━━━━━━━━━━━━ 20s 54ms/step - accuracy: 0.0380 - loss: 4.1066 - val_accuracy: 0.1138 - val_loss: 3.5899
Epoch 2/50
236/236 ━━━━━━━━━━━━━━━━━━━━ 7s 28ms/step - accuracy: 0.1095 - loss: 3.5931 - val_accuracy: 0.1961 - val_loss: 3.1802
Epoch 3/50
236/236 ━━━━━━━━━━━━━━━━━━━━ 6s 27ms/step - accuracy: 0.1603 - loss: 3.3006 - val_accuracy: 0.2616 - val_loss: 2.8926
Epoch 4/50
236/236 ━━━━━━━━━━━━━━━━━━━━ 7s 29ms/step - accuracy: 0.2045 - loss: 3.0756 - val_accura